# Run H: LoRA with the 'no fallacy' class (same settings as run B)
Runtime → Change runtime type → **T4 GPU**.

Data comes from the private Hugging Face dataset `sreejithb/fallacy-dataset` (version `v3`, which adds the no-fallacy files).

**One-time Colab setup:** left sidebar → key icon (Secrets) → add secret `HF_TOKEN` (a Hugging Face **read** token) and switch on *Notebook access*. Upload `fallacy_code.zip` to the Files sidebar (code only, tiny).

Run H uses exactly run B's settings (rank 32, 5 epochs, lr 2e-4, batch 8, no validation holdout). The only difference is the extra 300 `Fallacy: None` training examples.

In [ ]:
!nvidia-smi -L
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate tqdm huggingface_hub

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()  # the secret can only be read from the notebook process, not from a !python subprocess
!rm -rf /content/work && mkdir -p /content/work && unzip -oq /content/fallacy_code.zip -d /content/work
%cd /content/work
!python fetch_data.py --rev v3
!wc -l data/generated/*.jsonl

## Train run H (expect `2499 training examples`: 1,222 + 977 + 300)

In [ ]:
!python train.py --out out/lora_H --epochs 5 --rank 32 --val-per-label 0 2>&1 | tee train_H.log

## Evaluate run H on the fallacy eval set (accuracy must not drop much from B: 74.1% strict) and on the no-fallacy eval set

In [ ]:
!python evaluate.py --adapter out/lora_H --out lora_H_preds.jsonl

In [ ]:
!python evaluate.py --adapter out/lora_H --eval-file data/generated/eval_none.jsonl --out none_H_preds.jsonl

## Top-k on the fallacy eval set (candidates now include `None`)

In [ ]:
!python topk.py --adapter out/lora_H --run H

In [ ]:
!zip -qr lora_result_H.zip out data/lora_H_preds.jsonl data/none_H_preds.jsonl data/topk_H.json train_H.log
from google.colab import files
files.download('/content/work/lora_result_H.zip')